# Encontro 09 - Análise de Sensibilidade

In [ ]:
# Importa as bibliotecas necessárias
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

## Passo 1: Carregue os dados

Para os fins desta instrução, os dados são obtidos por meio de um link (não depende de um Google Drive em particular).

No contexto do projeto, os dados são fornecidos pelo Google Analytics do parceiro.

In [ ]:
# Baixa o dataset
!gdown 1eXYayriTnBDxbM_NBHF62ybTGK5dOFDh

Downloading...
From: https://drive.google.com/uc?id=1eXYayriTnBDxbM_NBHF62ybTGK5dOFDh
To: /content/dataset05-análise-de-sensibilidade.csv
100% 24.1k/24.1k [00:00<00:00, 63.8MB/s]


In [ ]:
# No projeto, esses dados viriam do Google Analytics
df = pd.read_csv('/content/dataset05-análise-de-sensibilidade.csv')
df["Data"] = pd.to_datetime(df["Data"])
df.head(3)

,Data,Taxa de Rejeição (%),Taxa de Conversão (%),Tempo Médio de Sessão (minutos)
0,2023-01-01,51.490142,1.638902,5.391691
1,2023-01-02,49.516526,2.215419,3.065232
2,2023-01-03,51.805703,2.038806,5.860462


## Passo 2: Explore as relações entre as variáveis

Realize uma exploração profunda dos dados.

### 2.1 Tempo de Sessão vs Taxa de Conversão

In [ ]:
fig = px.scatter(
    df,
    x="Tempo Médio de Sessão (minutos)",
    y="Taxa de Conversão (%)",
    trendline="ols",  # Adiciona uma linha de regressão linear ordinária ('ordinary least squares')
    title="Relação entre Tempo de Sessão e Taxa de Conversão",
    labels={"Tempo Médio de Sessão (minutos)": "Tempo Médio de Sessão (min)",
            "Taxa de Conversão (%)": "Taxa de Conversão (%)"}
)
fig.update_layout(
    xaxis_title="Tempo Médio de Sessão (min)",
    yaxis_title="Taxa de Conversão (%)",
    height=500,
    width=800
)
fig.show()

### 2.2 Taxa de Rejeição vs Taxa de Conversão

In [ ]:
fig = px.scatter(
    df,
    x="Taxa de Rejeição (%)",
    y="Taxa de Conversão (%)",
    trendline="ols",  # Adiciona uma linha de regressão linear ('ordinary least squares')
    title="Relação entre Taxa de Rejeição e Taxa de Conversão",
    labels={"Taxa de Rejeição (%)": "Taxa de Rejeição (%)",
            "Taxa de Conversão (%)": "Taxa de Conversão (%)"}
)
fig.update_layout(
    xaxis_title="Taxa de Rejeição (%)",
    yaxis_title="Taxa de Conversão (%)",
    height=500,
    width=800
)
fig.show()

### 2.3 Análise das correlações

In [ ]:
# Cria um dataset filtrado somente com as variáveis que interessam para a correlação
corr_df = df.filter(['Taxa de Rejeição (%)', 'Taxa de Conversão (%)', 'Tempo Médio de Sessão (minutos)'])
print(corr_df.corr('spearman'))
print("*" * 80)
print(corr_df.corr('pearson'))

                                 Taxa de Rejeição (%)  Taxa de Conversão (%)  \
Taxa de Rejeição (%)                         1.000000              -0.801434   
Taxa de Conversão (%)                       -0.801434               1.000000   
Tempo Médio de Sessão (minutos)             -0.706182               0.680363   

                                 Tempo Médio de Sessão (minutos)  
Taxa de Rejeição (%)                                   -0.706182  
Taxa de Conversão (%)                                   0.680363  
Tempo Médio de Sessão (minutos)                         1.000000  
********************************************************************************
                                 Taxa de Rejeição (%)  Taxa de Conversão (%)  \
Taxa de Rejeição (%)                         1.000000              -0.791121   
Taxa de Conversão (%)                       -0.791121               1.000000   
Tempo Médio de Sessão (minutos)             -0.696426               0.672140   

         

> *A falta de correlação impede a análise de sensibilidade? Não!*

### Passo 3: Crie os modelos preditivos

#### Passo 3.1 Variáveis linearmente relacionadas

##### Passo 3.1.1 Taxa de Conversão vs Tempo de Sessão

O modelo preditivo  da Taxa de Conversão (%) baseando-se no Tempo de Sessão é dado por:

$$
C = 0,37787 \times S + 0,969023
$$

onde $C$ é a taxa de conversão (%) e $S$, o tempo de sessão (minutos).

##### Passo 3.1.2 Taxa de Conversão vs Tempo de Sessão

O modelo preditivo da taxa de conversão baseando-se na taxa de rejeição (%) é dado por:

$$
C = -0,176168 \times R + 11,0774
$$

onde $C$ é a taxa de conversão (%) e $R$, a taxa de rejeição (%).

#### Passo 3.2 Variáveis não linearmente relacionadas

##### 3.2.1 Treinamento de modelo de machine learning

Quando não conseguimos determinar a natureza das relações entre as variáveis, podemos treinar um modelo de machine learning para fazer o trabalho pesado. Vamos treinar um modelo de florestas aleatórias.

In [ ]:
# Removendo a coluna de data para focar nas variáveis numéricas
X = df.drop(["Data", "Taxa de Conversão (%)"], axis=1)
y = df["Taxa de Conversão (%)"]

# Dividindo os dados em conjuntos de treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Criando e treinando o modelo de floresta aleatória
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Avaliando o modelo
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {np.sqrt(mse):.2f}")
print(f"MAE: {mae:.2f}")

RMSE: 1.15
MAE: 0.93


**RMSE:**

$1.15$

**Interpretação**: O RMSE de 1.15 indica que, tipicamente, as previsões do modelo desviam da realidade (tanto para mais quanto para menos) em cerca de 1.15%. Este valor proporciona uma ideia da magnitude média dos erros nas previsões do modelo, com todos os erros sendo ponderados igualmente e os maiores erros tendo uma influência proporcionalmente maior devido ao quadrado. Um RMSE de 1.15% sugere que as previsões têm um desvio padrão de 1.15% em torno dos valores reais.

**MAE:**

$0.93$

**Interpretação**: O MAE de 0.93% significa que o erro absoluto médio das previsões do modelo é de 0.93%. Isso reflete que, em média, as previsões erram por menos de 1% em relação aos valores reais. Este valor é um indicador de erro mais direto e menos sensível a outliers do que o RMSE, indicando que, de forma geral, o modelo possui uma precisão razoável na previsão das taxas.

**CONCLUSÃO**

Comparando RMSE e MAE, notamos que o RMSE é um pouco maior que o MAE, o que é comum e sugere a presença de alguns erros maiores nas previsões (já que o RMSE eleva os erros ao quadrado, aumentando o impacto de erros maiores na média). No entanto, ambos os valores estão relativamente próximos, o que implica que não há muitos erros extremamente grandes distorcendo a avaliação de erro médio do modelo.


## Passo 4: Realize a análise de sensibilidade

### 4.1 Variações linearmente relacionadas

#### 4.1.1 Análise de Impacto de Alteração no Tempo de Sessão

Calcule o impacto no percentual da taxa de conversão se o tempo médio de sessão aumentar de 5 para 6 minutos.

**MODELO:** $ C = 0,37787 \times S + 0,969023 $

*Faça os cálculos e escreva sua análise aqui*

**CÁLCULO**:
- **Taxa de Conversão para S=5 minutos**:
  $$
  C = 0,37787 \times 5 + 0,969023 = 1,889435 + 0,969023 = 2,858458\%
  $$
- **Taxa de Conversão para S=6 minutos**:
  $$
  C = 0,37787 \times 6 + 0,969023 = 2,267422 + 0,969023 = 3,236445\%
  $$

**ANÁLISE**: Aumentar o tempo de sessão de 5 para 6 minutos deve aumentar a taxa de conversão de aproximadamente 2,86% para 3,24%.

*Clique aqui para ver a resposta*

<!--
**CÁLCULO**:
- **Taxa de Conversão para S=5 minutos**:
  $$
  C = 0,37787 \times 5 + 0,969023 = 1,889435 + 0,969023 = 2,858458\%
  $$
- **Taxa de Conversão para S=6 minutos**:
  $$
  C = 0,37787 \times 6 + 0,969023 = 2,267422 + 0,969023 = 3,236445\%
  $$

**ANÁLISE**: Aumentar o tempo de sessão de 5 para 6 minutos deve aumentar a taxa de conversão de aproximadamente 2,86% para 3,24%.
-->

---

#### 4.1.2 Exploração da Sensibilidade do Modelo

Avalie como uma redução no tempo médio de sessão para 4 minutos afetaria a taxa de conversão. Utilize o modelo linear fornecido para prever o novo valor da taxa de conversão.

**MODELO:** $ C = 0,37787 \times S + 0,969023 $

*Faça os cálculos e escreva sua análise aqui*

**CÁLCULOS**:
- **Taxa de Conversão para S=4 minutos**:
  $$
  C = 0,37787 \times 4 + 0,969023 = 1,51148 + 0,969023 = 2,480503\%
  $$

**ANÁLISE**: Reduzir o tempo médio de sessão para 4 minutos deve diminuir a taxa de conversão para aproximadamente 2,48%.

*Clique aqui para ver a resposta*

<!--
**CÁLCULOS**:
- **Taxa de Conversão para S=4 minutos**:
  $$
  C = 0,37787 \times 4 + 0,969023 = 1,51148 + 0,969023 = 2,480503\%
  $$

**ANÁLISE**: Reduzir o tempo médio de sessão para 4 minutos deve diminuir a taxa de conversão para aproximadamente 2,48%.
-->

---

#### 4.1.3 Análise de Cenários Múltiplos

Usando o modelo linear, analise o impacto de aumentar o tempo médio de sessão de 5 minutos para 7 minutos e depois para 10 minutos. Compare esses cenários para ver como aumentos mais significativos no tempo médio de sessão influenciam a taxa de conversão.

**MODELO:** $ C = 0,37787 \times S + 0,969023 $

*Faça os cálculos e escreva sua análise aqui*

**MODELO**: $ C = 0,37787 \times S + 0,969023 $

**CÁLCULOS**:
- **Taxa de Conversão para S=7 minutos**:
  $$
  C = 0,37787 \times 7 + 0,969023 = 2,64509 + 0,969023 = 3,614113\%
  $$
- **Taxa de Conversão para S=10 minutos**:
  $$
  C = 0,37787 \times 10 + 0,969023 = 3,7787 + 0,969023 = 4,747723\%
  $$

**ANÁLISE**: Aumentar o tempo médio de sessão para 7 e 10 minutos aumenta a taxa de conversão para aproximadamente 3,61% e 4,75%, respectivamente.

*Clique aqui para ver a resposta*

<!--
**MODELO**: $ C = 0,37787 \times S + 0,969023 $

**CÁLCULOS**:
- **Taxa de Conversão para S=7 minutos**:
  $$
  C = 0,37787 \times 7 + 0,969023 = 2,64509 + 0,969023 = 3,614113\%
  $$
- **Taxa de Conversão para S=10 minutos**:
  $$
  C = 0,37787 \times 10 + 0,969023 = 3,7787 + 0,969023 = 4,747723\%
  $$

**ANÁLISE**: Aumentar o tempo médio de sessão para 7 e 10 minutos aumenta a taxa de conversão para aproximadamente 3,61% e 4,75%, respectivamente.
-->

#### 4.1.4 Estimativa de Impacto Combinado com Alteração na Taxa de Rejeição

Suponha que melhorias no conteúdo do site reduziram a taxa de rejeição de 50% para 45%. Usando o modelo linear, estime o efeito combinado dessa redução com um aumento no tempo médio de sessão de 5 para 6 minutos na taxa de conversão.

**MODELO**: $ C = -0,176168 \times R + 11,0774 $

*Faça os cálculos e escreva sua análise aqui*

*Clique aqui para ver a resposta*

<!--
**CÁLCULOS**:
- **Taxa de Conversão para R=45%**:
  $$
  C = -0,176168 \times 45 + 11,0774 = -7,92756 + 11,0774 = 3,14984\%
  $$

**ANÁLISE**: Reduzir a taxa de rejeição de 50% para 45% aumenta a taxa de conversão para aproximadamente 3,15%.
-->

---

#### 4.1.5 Modelagem de Previsão a Longo Prazo

Preveja a taxa de conversão para os próximos 12 meses, assumindo um aumento gradual no tempo médio de sessão de 1 minuto por mês, começando de 5 minutos.

In [ ]:
# Faça sua implementação aqui

*Clique aqui para ver a resposta*

<!--

Em uma célula de código.

# Definição dos parâmetros do modelo
coef_tempo_sessao = 0.37787
intercepto = 0.969023

# Intervalo de tempo de sessão de 5 a 16 minutos
tempos_de_sessao = range(5, 17)

# Calculando a taxa de conversão para cada tempo de sessão
for tempo in tempos_de_sessao:
    taxa_de_conversao = coef_tempo_sessao * tempo + intercepto
    print(f"Tempo de Sessão: {tempo} minutos, Taxa de Conversão: {taxa_de_conversao:.2f}%")

Em uma célula de texto:
**ANÁLISE**: Assumindo que a relação linear se mantenha, ao aumentar o tempo de sessão para de 5 para 16 minutos, a taxa de conversão deve aumentar de 2,86% para cerca de 7%.
-->

### 4.2 Variáveis não linearmente relacionadas

#### 4.2.1 Análise de Impacto da Taxa de Conversão

In [ ]:
import pandas as pd
import numpy as np

# Suponha que estas sejam as novas taxas de rejeição e tempos de sessão que você quer testar
sensibilidade_df = pd.DataFrame({
    'Taxa de Rejeição (%)': np.linspace(40, 60, 5),  # De 40% a 60% em 5 passos
    'Tempo Médio de Sessão (minutos)': np.linspace(1, 10, 5)  # De 1 a 10 minutos em 5 passos
})

# Previsões usando o modelo treinado
predicted_conversions = model.predict(sensibilidade_df)

# Adicionando as previsões de volta aos dados de teste para análise
sensibilidade_df['Taxa de Conversão Prevista (%)'] = predicted_conversions

sensibilidade_df.head(10)

,Taxa de Rejeição (%),Tempo Médio de Sessão (minutos),Taxa de Conversão Prevista (%)
0,40.0,1.00,2.920912
1,45.0,3.25,3.207527
2,50.0,5.50,2.585694
3,55.0,7.75,2.723062
4,60.0,10.00,2.129324


#### 4.2.2 Simulação de Monte Carlo

In [ ]:
# Configurações do modelo
n_simulations = 1000
predicted_conversions = []

for _ in range(n_simulations):
    # O método de bootstrapping nesta linha de código cria uma nova amostra dos dados originais,
    # permitindo repetições, para estimar a variabilidade das estatísticas ou do modelo
    bootstrap_sample = resample(X, n_samples=len(X), replace=True, random_state=None)

    # Fazendo previsões com o modelo de floresta aleatória
    predicted_conversion = model.predict(bootstrap_sample)

    # Armazene os resultados
    predicted_conversions.extend(predicted_conversion)

# Converta os resultados em um DataFrame para análise
predicted_conversions_df = pd.DataFrame(predicted_conversions, columns=['Taxa de Conversão Prevista (%)'])

# Calculando estatísticas descritivas
statistics_df = predicted_conversions_df.describe()

print(statistics_df)

       Taxa de Conversão Prevista (%)
count                   365000.000000
mean                         4.481817
std                          1.496679
min                          1.258401
25%                          3.150568
50%                          4.476099
75%                          5.731760
max                          7.438228


In [ ]:
# Criando o histograma
fig = px.histogram(predicted_conversions_df, x='Taxa de Conversão Prevista (%)',
                   title='Distribuição da Taxa de Conversão Prevista',
                   labels={'Taxa de Conversão Prevista (%)': 'Taxa de Conversão (%)'},
                   nbins=30,
                   opacity=0.75,  # Opacidade das barras
                   color_discrete_sequence=['indianred'])  # Cor das barras

fig.update_layout(
    xaxis_title='Taxa de Conversão (%)',
    yaxis_title='Contagem',
    bargap=0.2,  # Espaçamento entre as barras
    width=800,
    height=500
)
fig.show()